# Análisis Zero-Shot de Clasificación de Pólipos

Este cuaderno analiza la salida del `batch_runner.py` para el experimento zero-shot.
Se evalúa la calidad de clasificación (`predicted_class`) frente al ground truth (`ground_truth_cls`)
y se revisan métricas de rendimiento del modelo (TTFT y TPS).

## 1) Configurar entorno y dependencias

Si faltan paquetes, ejecuta: `uv add scikit-learn seaborn`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)

## 2) Definir parámetros y constantes iniciales

Ajusta estas rutas según tu ejecución más reciente.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "batch_results" / "batch_PolypClassificationWithReasoning_20260312_193456.jsonl"
MANIFEST_PATH = PROJECT_ROOT / "data" / "experiments" / "zeroshot_manifest.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "batch_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_CLASSES = ["AD", "HP", "ASS", "UNKNOWN"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"Manifest path: {MANIFEST_PATH}")

## 3) Implementar funciones base

Estas funciones encapsulan la carga robusta, parseo del `payload` y resolución de ground truth.

In [ ]:
def load_table(path: Path) -> pd.DataFrame:
    """Load CSV or JSONL batch/manifest files into a DataFrame."""
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".jsonl":
        rows: list[dict[str, Any]] = []
        with path.open("r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return pd.DataFrame(rows)

    raise ValueError(f"Unsupported file extension: {suffix}")


def parse_payload(value: Any) -> dict[str, Any]:
    """Parse payload if it is already a dict or a JSON string."""
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return {}
        try:
            parsed = json.loads(text)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}


def resolve_ground_truth(results_df: pd.DataFrame, manifest_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """Prefer in-row ground truth and fallback to manifest merge by image_id or image_path."""
    df = results_df.copy()
    if "ground_truth_cls" not in df.columns:
        df["ground_truth_cls"] = np.nan

    needs_fallback = df["ground_truth_cls"].isna() | (df["ground_truth_cls"].astype(str).str.strip() == "")

    if manifest_df is None or manifest_df.empty or not needs_fallback.any():
        return df

    manifest = manifest_df.copy()
    for col in ["image_id", "image_path", "ground_truth_cls"]:
        if col not in manifest.columns:
            manifest[col] = np.nan

    fallback_cols = ["image_id", "image_path"]
    fallback_map = manifest[fallback_cols + ["ground_truth_cls"]].dropna(subset=["ground_truth_cls"])

    by_id = fallback_map.dropna(subset=["image_id"]).drop_duplicates(subset=["image_id"]).set_index("image_id")["ground_truth_cls"]
    by_path = fallback_map.dropna(subset=["image_path"]).drop_duplicates(subset=["image_path"]).set_index("image_path")["ground_truth_cls"]

    missing_idx = df.index[needs_fallback]
    for idx in missing_idx:
        image_id = df.at[idx, "image_id"] if "image_id" in df.columns else np.nan
        image_path = df.at[idx, "image_path"] if "image_path" in df.columns else np.nan

        if pd.notna(image_id) and image_id in by_id.index:
            df.at[idx, "ground_truth_cls"] = by_id.loc[image_id]
        elif pd.notna(image_path) and image_path in by_path.index:
            df.at[idx, "ground_truth_cls"] = by_path.loc[image_path]

    return df

## 4) Construir el flujo principal de ejecución

Se carga la salida del batch, se extrae `predicted_class` y se alinea con el ground truth.

In [ ]:
results_df = load_table(RESULTS_PATH)
manifest_df = load_table(MANIFEST_PATH) if MANIFEST_PATH.exists() else None

if "payload" in results_df.columns:
    results_df["payload_parsed"] = results_df["payload"].apply(parse_payload)
else:
    results_df["payload_parsed"] = [{} for _ in range(len(results_df))]

results_df["predicted_class"] = results_df["payload_parsed"].apply(lambda d: d.get("predicted_class"))

eval_df = resolve_ground_truth(results_df, manifest_df=manifest_df)

if "status" in eval_df.columns:
    eval_df = eval_df[eval_df["status"] == "ok"].copy()

eval_df["predicted_class"] = eval_df["predicted_class"].astype(str).str.upper()
eval_df["ground_truth_cls"] = eval_df["ground_truth_cls"].astype(str).str.upper()
eval_df = eval_df[
    eval_df["predicted_class"].isin(VALID_CLASSES)
    & eval_df["ground_truth_cls"].isin(VALID_CLASSES)
].copy()

print(f"Rows in raw results: {len(results_df)}")
print(f"Rows used for evaluation: {len(eval_df)}")
eval_df[["image_id", "image_path", "ground_truth_cls", "predicted_class"]].head(10)

## 5) Agregar validaciones y pruebas rápidas

Se comprueba consistencia mínima antes de calcular métricas.

In [ ]:
assert len(eval_df) > 0, "No hay filas válidas para evaluación. Revisa filtros, status o rutas."
assert eval_df["predicted_class"].isin(VALID_CLASSES).all(), "Hay clases predichas fuera del conjunto esperado."
assert eval_df["ground_truth_cls"].isin(VALID_CLASSES).all(), "Hay etiquetas GT fuera del conjunto esperado."

y_true = eval_df["ground_truth_cls"]
y_pred = eval_df["predicted_class"]

accuracy = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=VALID_CLASSES, average="macro", zero_division=0
)
precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=VALID_CLASSES, average="micro", zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=VALID_CLASSES, average="weighted", zero_division=0
)

metrics_df = pd.DataFrame(
    [
        {"metric": "accuracy", "value": accuracy},
        {"metric": "precision_macro", "value": precision_macro},
        {"metric": "recall_macro", "value": recall_macro},
        {"metric": "f1_macro", "value": f1_macro},
        {"metric": "precision_micro", "value": precision_micro},
        {"metric": "recall_micro", "value": recall_micro},
        {"metric": "f1_micro", "value": f1_micro},
        {"metric": "precision_weighted", "value": precision_weighted},
        {"metric": "recall_weighted", "value": recall_weighted},
        {"metric": "f1_weighted", "value": f1_weighted},
    ]
)

report_dict = classification_report(
    y_true, y_pred, labels=VALID_CLASSES, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report_dict).T

display(metrics_df)
display(report_df)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=VALID_CLASSES)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    xticklabels=VALID_CLASSES,
    yticklabels=VALID_CLASSES,
    cbar=True,
)
plt.title("Matriz de confusión - Zero-Shot Polyp Classification")
plt.xlabel("Predicción")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.show()

In [ ]:
perf_cols = ["ttft_seconds", "tokens_per_second"]
available_perf = [col for col in perf_cols if col in eval_df.columns]

if available_perf:
    fig, axes = plt.subplots(1, len(available_perf), figsize=(6 * len(available_perf), 4))
    if len(available_perf) == 1:
        axes = [axes]

    for ax, col in zip(axes, available_perf):
        sns.histplot(eval_df[col].dropna(), kde=True, bins=25, ax=ax, color="#2a9d8f")
        ax.set_title(f"Distribución de {col}")
        ax.set_xlabel(col)

    plt.tight_layout()
    plt.show()

    if "predicted_class" in eval_df.columns:
        for col in available_perf:
            plt.figure(figsize=(7, 4))
            sns.boxplot(data=eval_df, x="predicted_class", y=col, order=VALID_CLASSES, color="#8ecae6")
            plt.title(f"{col} por clase predicha")
            plt.xlabel("Clase predicha")
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()
else:
    print("No se encontraron columnas TTFT/TPS en los resultados.")

## 6) Registrar salida y guardar resultados

Se guardan métricas agregadas y reporte por clase para trazabilidad del experimento.

In [ ]:
metrics_output_path = OUTPUT_DIR / "zeroshot_metrics_summary.csv"
report_output_path = OUTPUT_DIR / "zeroshot_classification_report.csv"
eval_output_path = OUTPUT_DIR / "zeroshot_eval_table.csv"

metrics_df.to_csv(metrics_output_path, index=False)
report_df.to_csv(report_output_path, index=True)
eval_df.to_csv(eval_output_path, index=False)

print(f"Saved: {metrics_output_path}")
print(f"Saved: {report_output_path}")
print(f"Saved: {eval_output_path}")